# Winner public CatBoost baseline
[Konstantin Yakovlev 공개 학습 코드](https://www.kaggle.com/kyakovlev/ieee-catboost-baseline-with-groupkfold-cv)를 실제 출처로 사용한다.
[원본 CSV minification](https://www.kaggle.com/kyakovlev/ieee-data-minification) → [공개 FE](https://www.kaggle.com/kyakovlev/ieee-fe-with-some-eda) → 월별 GroupKFold 학습 경로다.
최종 우승 CAT/LGB 모델 전체 코드로 확인된 것은 아닌 **우승자 본인의 공개 baseline**이다. 17위 등 다른 참가자 구현을 사용하지 않는다.
공통 FE는 `data/winner_fe.py`에 분리했고 학습은 이 노트북에서 직접 한다. 전처리 pickle이나 EDA 실행을 선행할 필요가 없다.

## 이 노트북을 읽는 방법

코드를 실행하기 전에 바로 위의 설명을 읽어보세요. **어떤 질문을 푸는지 → 작은 예시로 계산 → 실제 코드의 변수와 연결 → 출력 해석** 순서로 설명합니다.
코드 아래의 관찰은 이미 저장된 실행 결과를 읽는 안내입니다. '해볼 실험'은 아직 실행하지 않은 제안이며, 실제 결과와 구분했습니다.

처음 읽을 때 함수 이름을 모두 외울 필요는 없습니다. 새 피처를 만날 때마다 **'이 숫자는 무엇을 요약하며, 예측할 때도 알 수 있는가?'**를 물어보세요.
EDA에서 찾은 차이가 모델 성능 개선을 뜻하지는 않습니다. 실험 노트북에서는 **검증 데이터를 정한 뒤 구성 요소 하나씩 비교**해야 개선의 근거를 얻습니다.

처음에는 `01_eda_report` → GitHub `baseline`을 읽고, 이후 `02_winner_eda` → `winner_xgb` → `winner_lgbm` → `winner_catboost` → `winner_blend`로 이어가세요.
우승자 공개 baseline과 최종 우승 제출 전체는 구분합니다.

설명을 보강하면서 학습 코드·기존 표·그래프·실행 범위를 유지했습니다. 이 노트북의 **저장된 출력**과 설정의 **다음 실행 기본값**이 다를 수 있으므로 첫 실행 범위와 metrics를 먼저 확인하세요.


### 처음 만나는 용어는 여기서 잠깐 확인하세요

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 피처(feature) | 모델에 입력할 거래의 정보. 원본 열과 새로 계산한 열 모두 포함 |
| NaN / 결측 | 값이 관측되지 않음. 실제 숫자 0과 다른 상태 |
| fold / validation | 교차검증의 한 분할 / 그 분할에서 평가용으로 제외한 데이터 |
| OOF | 각 train 행을 그 행 없이 학습한 모델로 예측해서 모은 값 |
| smoke | 전체 학습 전에 축소 데이터·rounds로 실행 흐름을 확인하는 실험 |
| leaf | tree에서 조건을 따라 내려간 끝의 구역. 그 구역에 들어온 행에 같은 보정을 줌 |

**제거 실험(ablation)**은 피처나 기법 하나를 뺀 모델을 같은 검증에서 비교하는 방법입니다. '있을 때 좋았으니 도움이 된다'에서 한 걸음 더 나아가 실제 기여를 확인하려는 실험입니다.

## 1. 원본 CSV와 공개 FE

### 이 실험에서 배울 것은 '범주를 어떻게 활용할까?'입니다

사기를 가르는 단서는 금액처럼 크기를 비교하는 수치일 수도 있고, 기기·카드 조합처럼 '무엇인가'가 중요한 범주일 수도 있습니다.
CatBoost는 범주형 정보를 활용하는 tree boosting입니다. 하지만 모델 이름만 바꾸는 실험은 아닙니다. 저자의 공개 CatBoost baseline은 **공통 FE → CatBoost용 추가 가공 → 월별 검증**을 사용합니다.

아래 `NROWS`가 없으면 전체 원본 CSV를 읽습니다. 값이 있으면 각 월에서 일정 비율로 표본을 뽑아 실행 흐름을 확인합니다.
앞부분만 읽는 Model2 smoke와 표본 방식이 다릅니다. `TAG`는 산출물 폴더 이름이고, 저장된 로그에서 실제 표본과 rounds를 확인할 수 있습니다.
`target_keys`는 뒤에서 target mean을 만들 때 사용할 ProductCD/M4의 현재 그룹 키를 복사합니다. 전처리 이후 열이 바뀌더라도 fold별 같은 키로 라벨 평균을 계산하려는 목적입니다.

`build_features`는 한 줄이지만 많은 가공을 합니다. 아래 설명을 읽고 함수 안의 각 단계를 찾아보세요. **어떤 정보를 추가하는지와 계산 순서**가 핵심입니다.

### `build_features()` 안에서 만드는 피처도 여기서 공부합니다

두 모델이 같은 FE를 쓰므로 구현만 `data/winner_fe.py`에 모았습니다. 코드 아래의 import를 몰라도 흐름을 이해할 수 있도록 실제 단계와 예시를 짚어봅니다.

**1. 원본 CSV와 값의 표현.** train/test transaction과 identity를 읽고 ID로 결합합니다. card4/card6/ProductCD/M4는 공동 빈도로 바꾸고 M의 T/F는 1/0으로 바꿉니다.
범주 A/B/C가 각각 100/10/10번이면 빈도는 흔함을 표현하지만 B/C는 같은 10이 됩니다. **빈도는 범주 정체성의 완전한 대체가 아닙니다.**
test의 `isFraud=0`은 표 구조를 맞추는 임시 값입니다. 실제 정답도, 정상 거래라는 판정도 아닙니다. 모델 입력에서 제외하고 학습 정답은 train에서만 가져옵니다.
float32는 메모리를 줄이는 수치 표현입니다. 표준화나 새로운 예측 정보가 아니며, original float16 축소보다 정밀도를 더 유지하는 변경입니다.

**2. 시간과 활동 비율.** 월/주/일/시간과 휴일 표시를 만든 뒤 `DT_M/DT_W/DT_D`는 집계 기준으로 쓰고 모델 입력에서 제외합니다.
어느 은행 그룹의 하루 거래가 100건이라도 그날 전체가 200건인지 20,000건인지에 따라 의미가 다릅니다. `time_frequency`는 **그룹 거래 수 / 그 시간 블록 전체 거래 수**를 만듭니다.
은행 그룹의 평균/최빈 활동 시간에서 현재 시간을 뺀 피처도 만듭니다. 평소 오후인데 새벽인 거래를 표현할 후보입니다. 시계의 23시와 0시가 실제로는 가까운 점은 단순 차이가 잘 표현하지 못합니다.

**3. 희귀 카드 처리와 여러 UID.** 원문대로 train/test 양쪽에 존재하지 않는 카드 코드를 결측으로 만들고 card1 중 2건 이하도 비웁니다.
이는 드문 식별자를 암기하는 일을 줄이려는 선택입니다. 반면 희귀한 카드의 정보도 잃으므로 항상 이득인 규칙은 아닙니다. test 전체 코드 목록을 볼 수 있는 대회 배치 조건을 사용합니다.
`uid→uid2→uid3→uid4/5`는 카드, 추가 카드 속성, 주소, 이메일을 점점 더 붙입니다. 좁게 묶으면 개인화되지만 표본이 줄고, 넓게 묶으면 통계가 안정적인 대신 여러 고객이 섞입니다.

**4. 그룹 평균과 변동.** 같은 UID의 금액 [10,10,40]의 평균은 20이고 표본 std는 약 17.32입니다.
이 std는 `sqrt(((10-20)²+(10-20)²+(40-20)²)/(3-1))`입니다. 평균에서 얼마나 벗어나는지 제곱으로 모아 표본 변동을 요약합니다.
현재 금액 40과 평균 20을 같이 주면 모델이 '이 그룹에서 큰 금액인가'를 배울 수 있습니다. std는 활동의 다양성을 표현합니다. 1건 그룹의 std는 0이 아니라 NaN입니다.
`aggregate`는 D와 금액의 mean/std를 card/UID/bank 그룹에 붙입니다. **정답 평균이 아니라 입력 피처의 평균**이므로 target encoding과 다릅니다.

**5. 같은 시기 안에서의 정규화.** `normalize`는 train과 test 각각의 일/주/월 안에서 min-max와 표준 점수를 만듭니다.
같은 기간의 값 [10,20,30]에서 30의 min-max는 (30-10)/(30-10)=1입니다. 평균 20, 표본 std 10이면 표준 점수는 (30-20)/10=1입니다.
같은 기간에서 상대적으로 높은 값을 표현하지만 이 값은 UID 안의 비교가 아닙니다. tree에 단순 스케일링이 필수라서가 아니라 **시간 블록에 따른 상대 위치라는 새 정보**를 만들기 위해 씁니다. 값이 모두 같으면 분모가 0이라 결측이 생길 수 있습니다.

**6. D와 금액/C 변환의 순서.** 앞의 D 그룹 집계는 clip 이전 값으로 만듭니다. 이후 D의 음수를 0으로 clip하고 D8/D9의 결측/소수 관계 피처를 만든 뒤 정규화합니다. D1/D2는 train 최대값으로 나눈 별도 scaled 열을 만듭니다. 원본 D를 자기 빈도 값으로 바꾸기 때문에 변환 이후 D 열은 '원래 일수'가 아닙니다.
금액은 5,000으로 clip하고 집계/정규화/상품×금액 빈도를 만든 후 `log1p`로 원본 금액을 바꿉니다. clip은 정보 일부를 버리고 log는 양수 범위 순서를 유지하며 큰 간격을 줄입니다.
C에는 빈도를 추가하고 마지막 train 월의 최대값으로 상한을 둡니다. 집계가 **원 단위 금액인지, log 금액인지**는 만들어진 순서에 달려 있습니다.

**7. 디바이스/ID와 문자열.** identity의 Found/T 같은 값을 매핑하고, 화면 해상도를 가로/세로로 나누며, 기기/OS/브라우저에서 문자와 숫자 부분을 분리합니다.
이를 빈도로 요약하면 희귀한 버전/기기를 표현할 수 있지만 정확한 버전 숫자가 서열을 의미하지는 않습니다. 남은 문자열은 공동 label encoding 후 category로 둡니다.
ProductCD/M4 target mean은 공통 FE에서 미리 만들지 않고 **각 모델의 fit fold 정답**으로 계산합니다.

이 FE는 여러 입력을 동시에 사용해 **전체 배치에서의 문맥**을 만듭니다. 정답이 없다고 자동으로 실시간에 쓸 수 있는 것은 아닙니다. 하루 전체 빈도나 미래 거래까지 포함한 평균은 그 거래 순간에는 아직 모를 수 있습니다.

In [1]:
import gc, json, logging, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'source.json').exists())
sys.path.insert(0, str(ROOT))
from data.loader import get_data_dir
from data.winner_fe import CAT_ORIGINAL, build_features
NROWS = int(os.environ['IEEE_WINNER_NROWS']) if os.getenv('IEEE_WINNER_NROWS') else None
FOLDS = int(os.getenv('IEEE_WINNER_FOLDS', '6'))
TAG = os.getenv('IEEE_WINNER_TAG', 'smoke' if NROWS else 'full')
OUT = ROOT / 'experiments/winner_catboost/outputs' / TAG
OUT.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger('winner_catboost')
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.addHandler(logging.FileHandler(OUT / 'run.log', mode='w', encoding='utf-8'))
logger.addHandler(logging.StreamHandler(sys.stdout))
started = time.perf_counter()
train, test, remove, original = build_features(NROWS)
target_keys = {col: (train[col].copy(), test[col].copy()) for col in ['ProductCD', 'M4']}
logger.info('FE train=%s test=%s monthly rows=%s', train.shape, test.shape, train.DT_M.value_counts().sort_index().to_dict())

FE train=(30000, 790) test=(30002, 790) monthly rows={12: 6976, 13: 4703, 14: 4370, 15: 5163, 16: 4250, 17: 4538}


## 2. 모델별 공개 코드의 준비 단계

### CatBoost에 넣기 전에, 저자는 정보를 어떻게 바꾸었을까요?

긴 셀이므로 순서대로 읽어보겠습니다. 같은 범주를 숫자로 표현하는 방식에도 서로 다른 뜻이 있습니다.

**① 결측을 하나의 단서로 만듭니다.** `missing`은 train+test에서 각 열의 NaN 개수를 셉니다.
NaN 개수가 같은 열들을 모아 `nan_group_*`를 만들고, 그중 하나라도 누락이면 1을 줍니다. A/B 두 열이 각각 100개씩 결측이라고 해서 **같은 100행**에서 빠진 것은 아닙니다.
따라서 이 그룹은 '결측 개수가 같은 열의 합친 누락 여부'입니다. 앞 EDA의 행별 결측 마스크 동일성 확인과 구분해야 합니다.

**② 일부 범주의 별도 표현을 복원합니다.** `_cat` 열은 `original`에서 거래 ID로 찾아 붙입니다.
여기서 original은 **minification 이후, 공통 FE 이전**의 값입니다. minification에서 이미 빈도 숫자로 바꾼 ProductCD/card4 등의 원래 문자열을 모두 되살리는 것은 아닙니다.
공통 FE로 추가 가공한 표현과 그 이전 표현을 함께 제공하려는 단계입니다. 완전히 같은 일부 train 열은 중복을 줄입니다.

**③ 85%를 넘는 지배값은 '그 값인가?'로 바꿉니다.** 예를 들어 100행 중 90행이 0이고 나머지가 2/7/20이면 `eq(0)`이 1/0 flag를 만듭니다.
드문 차이에 주목하는 간단한 표현이지만 2와 20의 차이는 사라집니다. NaN을 -999로 채운 뒤 그것이 지배값이면 **누락=1**인 flag가 됩니다. 모든 flag의 1이 '사기'나 '이상'을 뜻하지 않습니다.

**④ UID를 범주로 다시 포함합니다.** 공통 remove 목록에 있던 `uid`~`uid5`, `bank_type`을 이 실험은 되살립니다.
넓은 그룹과 좁은 그룹의 반복 패턴을 모델이 배울 수 있지만, 특정 그룹의 라벨을 기억할 위험도 커집니다. UID를 직접 빼고 집계 피처를 쓰는 XGB와 중요한 차이입니다.

**⑤ 실제로 같은 수치 열을 줄입니다.** `means`의 평균은 비교 후보를 빨리 찾는 열쇠이고, 삭제 결정은 `.equals()`로 합니다.
평균이 같다는 이유만으로 다른 열을 삭제하는 코드는 아닙니다. 이 구현의 후보 탐색은 완전한 모든 열 쌍 검사도 아닙니다.

**⑥ 범주를 정수 ID로 맞춥니다.** train/test를 같이 factorize하고 `cat_features`로 선언합니다.
숫자 0/1/2는 여기서 범주 이름표입니다. 이를 일반 수치 열로 넣어 '2가 1보다 크다'고 분기하는 것과 다릅니다. NaN의 -999도 범주 이름표로 처리됩니다.

### CatBoost의 범주 처리와 tree를 작은 예로 이해해봅시다

범주 A의 라벨이 [1,0,1]일 때 전체 평균 2/3을 세 행에 바로 붙이면 각 행의 자기 라벨도 피처에 들어갑니다.
CatBoost의 ordered target statistics는 순서를 정해 **그 행보다 앞선 행의 라벨**로 통계를 만드는 아이디어를 사용합니다.
설명용 prior=.5, prior 가중치=1인 예에서 첫 행은 .5, 두 번째는 `(1+.5)/(1+1)=.75`, 세 번째는 `(1+0+.5)/(2+1)=.5`입니다.
자기 정답을 그대로 포함하지 않으면서 반복 범주의 신호를 얻으려는 구조입니다. 실제 CTR 종류·prior·one-hot 선택은 라이브러리 설정에 따릅니다.
[공식 범주 통계 설명](https://catboost.ai/docs/en/concepts/algorithm-main-stages_cat-to-numberic)을 참고하세요.

기본 symmetric tree는 한 깊이의 모든 노드가 같은 분기 조건을 사용합니다. depth=8이면 최대 2⁸=256개 leaf가 생깁니다.
같은 leaf 수라도 LightGBM의 자유로운 leaf-wise 구조와 모양은 다릅니다. 이렇게 만든 tree를 순차적으로 더하며 Logloss를 줄입니다.
범주 통계의 ordered 아이디어와 `boosting_type='Ordered'`라는 학습 옵션은 별개입니다. 아래 params는 boosting_type을 지정하지 않아 **Ordered boosting을 항상 쓴다고 해석하지 않습니다.**
[학습 옵션 정의](https://catboost.ai/docs/en/references/training-parameters/common)에서 장치·데이터 크기에 따른 기본값을 확인할 수 있습니다.

| 설정 | 의미 | 읽을 때 주의할 점 |
|---|---|---|
| iterations=5000 | tree 수의 상한 | smoke 출력에서는 100으로 줄였음 |
| learning_rate=.07 | tree마다 보정하는 크기 | 작게 바꾸면 충분한 iterations도 필요 |
| depth=8 | symmetric tree 분기 깊이 | 더 깊으면 조합을 잘 외우지만 과적합/비용 증가 가능 |
| loss_function=Logloss | 실제 학습 목표 | eval_metric=AUC와 역할이 다름 |
| od_wait=500 | 개선 정체를 감시하는 대기 설정 | 100-round smoke에서 장기 정체를 검증할 수 없음 |
| task_type=GPU | 학습 장치 | 공개 baseline을 최종 우승 모델로 바꾸는 옵션은 아님 |

위 값은 공개 baseline 설정입니다. 최적값을 찾았다는 뜻이 아닙니다. 먼저 다음 셀의 검증 방식을 이해한 뒤 한 요소씩 비교하세요.

In [2]:
import catboost
from catboost import CatBoostClassifier

# Source cells 6–12: missing groups, original categories, domination, UID restore, duplicates.
missing = pd.concat([train.isna(), test.isna()]).sum()
categorical_features = []
for count, columns in missing.groupby(missing):
    names = list(columns.index)
    if count > 0 and len(names) > 1:
        name = f'nan_group_{int(count)}'
        for frame in [train, test]:
            frame[name] = frame[names].isna().any(axis=1).astype('int8')
        categorical_features.append(name)
for frame in [train, test]:
    restored = original.reindex(frame.TransactionID).reset_index(drop=True).add_suffix('_cat')
    restored.index = frame.index
    for col in restored:
        if pd.api.types.is_string_dtype(restored[col]):
            restored[col] = restored[col].astype('object')
        frame[col] = restored[col]
for col in CAT_ORIGINAL:
    # ProductCD/M4 are intentionally retained for fold-local target means.
    if col not in target_keys and train[col].equals(train[col + '_cat']):
        train.drop(columns=col, inplace=True); test.drop(columns=col, inplace=True)
    categorical_features.append(col + '_cat')
for col in list(train):
    if not isinstance(train[col].dtype, pd.CategoricalDtype):
        counts = train[col].fillna(-999).value_counts()
        if counts.iloc[0] / len(train) > .85 and col not in ['isFraud', 'C3_fq_enc']:
            dominant = counts.index[0]
            for frame in [train, test]:
                frame[col] = frame[col].fillna(-999).eq(dominant).astype('int8')
            categorical_features.append(col)
categorical_features += ['D8_not_same_day', 'TransactionAmt_check', 'uid', 'uid2', 'uid3', 'uid4', 'uid5', 'bank_type']
remove = [c for c in remove if c not in ['uid', 'uid2', 'uid3', 'uid4', 'uid5', 'bank_type']]
means = {}
for col in list(train.select_dtypes('number')):
    mean = train[col].to_numpy().mean()
    peers = means.setdefault(mean, [])
    if col not in target_keys and any(train[peer].equals(train[col]) for peer in peers):
        train.drop(columns=col, inplace=True); test.drop(columns=col, inplace=True)
    else:
        peers.append(col)
# Explicitly include existing category dtypes for current CatBoost.
categorical_features += list(train.select_dtypes('category'))
features = [c for c in train if c not in remove]
categorical_features = sorted(set(c for c in categorical_features if c in features))
for col in categorical_features:
    values, _ = pd.concat([train[col].astype('object').fillna(-999), test[col].astype('object').fillna(-999)]).factorize(sort=True)
    train[col], test[col] = values[:len(train)], values[len(train):]
train, test = train.copy(), test.copy()
ROUNDS = int(os.getenv('IEEE_WINNER_ROUNDS', '5000'))
params = dict(iterations=ROUNDS, learning_rate=.07, eval_metric='AUC', loss_function='Logloss',
              random_seed=42, metric_period=500, od_wait=500, task_type=os.getenv('IEEE_WINNER_CAT_DEVICE', 'GPU'), depth=8, thread_count=8)
logger.info('features=%d categorical=%d params=%s', len(features), len(categorical_features), params)

features=720 categorical=144 params={'iterations': 100, 'learning_rate': 0.07, 'eval_metric': 'AUC', 'loss_function': 'Logloss', 'random_seed': 42, 'metric_period': 500, 'od_wait': 500, 'task_type': 'GPU', 'depth': 8, 'thread_count': 8}


## 3. 월별 CV

### 정답을 쓰는 피처는 fold 안에서 다시 만들어야 합니다

모델이 범주를 잘 처리하더라도, 사람이 미리 만든 피처에서 validation 정답을 섞으면 이를 되돌릴 수 없습니다.
아래 `mapping`은 **fit_idx 라벨만** 사용하고 val/test에는 그 mapping을 적용합니다. target mean 처리와 CatBoost 내부 범주 통계는 서로 다른 단계입니다.

### '이 범주는 사기가 얼마나 많았는가'를 입력으로 쓰는 방법

Target mean encoding은 학습 데이터에서 범주별 정답 평균을 구해 피처로 넣습니다. 학습 fold에서 M4=A의 라벨이 [0,0,1,0]이면 값은 .25입니다.
검증 A도 .25를 받고, 학습에 없던 B는 그 fold 전체 사기율을 받습니다. **검증 B의 정답이 1이어도 인코딩을 만들 때 알면 안 됩니다.**

그래서 코드는 **fold 분리 → fit 정답으로 mapping 작성 → fit/validation/test에 같은 mapping 적용** 순서입니다.
전체 train 정답으로 mapping을 만들고 CV하면 검증 정답 일부가 이미 피처에 들어가 점수가 낙관적으로 보일 수 있습니다.

주의할 구분이 하나 더 있습니다. 이 구현은 validation 정답은 제외하지만 fit 행의 인코딩에는 자기 정답도 평균의 일부로 들어갑니다.
범주가 1건뿐이면 평균이 정답 자체라 과적합하기 쉽습니다. **높은 cardinality에서는 내부 OOF 인코딩/leave-one-out/전체 평균으로 smoothing을 고려할 수 있습니다.** 여기서는 그 추가 기법을 구현하거나 효과를 검증한 것은 아닙니다.

### fold, OOF, test 평균을 구분해서 읽기

Fold는 교차검증의 한 번의 학습/평가 분할입니다. 월별 6-fold이면 매번 한 월을 검증에 두고 나머지 월로 학습합니다.
검증 월의 행은 자기 정답을 학습하지 않은 모델에서 예측을 받습니다. 그 값을 원래 행 위치에 모은 것이 **OOF(out-of-fold)**입니다.
OOF를 합쳐 AUC를 구하면 학습 행 재예측보다 일반화 평가에 가깝습니다. 다만 여기서는 early stopping도 같은 validation을 보므로 완전히 손대지 않은 최종 평가셋은 아닙니다.

`pred += fold_test_pred / FOLDS`는 test를 각 fold 모델로 예측해 평균냅니다. test에는 정답이 없어 AUC를 계산할 수 없습니다.
GroupKFold의 groups는 월입니다. **같은 월의 행이 fit/validation 양쪽에 들어가지는 않지만 같은 UID의 다른 월 거래는 들어갈 수 있습니다.**
또 검증이 과거 월이면 이후 월도 fit에 들어갑니다. 그러므로 이 검증을 미래 예측이나 고객 전체 미관측 검증으로 부르면 안 됩니다.
미래 예측은 시간 holdout, 새 고객 예측은 UID 그룹 분할 등으로 따로 질문해야 합니다. [GroupKFold 정의](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)를 참고하세요.

### `oof`와 `oof_scaled`는 왜 둘 다 저장할까요?

`oof[val_idx] = values`는 해당 행을 학습에 넣지 않은 모델의 예측을 원래 위치에 돌려놓습니다. 모든 fold가 끝나면 모든 train 행에 OOF 예측 하나씩 생깁니다.
반면 `pred += ... / FOLDS`는 여러 모델이 **같은 test 행**에 낸 예측을 평균냅니다. OOF와 test 평균은 만드는 방식이 다릅니다.

저자의 공개 코드는 각 validation fold에서 예측을 `(p-min)/(max-min)`으로 바꾼 뒤 합친 OOF도 평가합니다. 이 노트북은 그 결과를 `oof_scaled`로 별도 보존합니다.
한 fold 안에서는 순서가 유지되므로 그 fold의 AUC는 그대로지만, **서로 다른 fold 사이의 순서**는 바뀔 수 있습니다.
예를 들어 fold A의 범위 [.7,.9]에서 .7은 0이 되고, fold B의 [.1,.2]에서 .2는 1이 됩니다. 원래 .7>.2였던 비교가 뒤집힙니다.

따라서 pooled AUC의 변화는 모델을 다시 학습해서 생긴 개선이 아닙니다. 이 변환은 실제 사기 빈도에 맞춘 확률 보정(calibration)도 아닙니다.
아래 코드는 test 예측에 같은 min-max를 적용하지 않습니다. **raw OOF, fold별 AUC, source-scaled OOF를 따로 읽고**, source-scaled 점수만으로 모델을 선택하지 마세요.

### 학습 루프의 나머지 줄도 목적을 연결해보세요

`replace([inf,-inf],nan)`은 0으로 나눈 비율 등이 만든 무한값을 결측으로 바꿉니다. 모든 누락을 임의의 금액 0으로 해석하는 처리는 아닙니다.
`cat_features`는 앞에서 만든 정수들이 범주 ID임을 모델에 알립니다. `use_best_model=True`는 validation 평가가 가장 좋았던 tree까지 보존합니다.
이 validation은 학습 행에는 들어가지 않지만 best iteration 선택에는 사용되므로 완전히 손대지 않은 최종 평가셋은 아닙니다.
마지막 두 assert는 누락된 OOF 행과 잘못된 확률 범위를 찾습니다. 좋은 AUC를 보장하는 검사와는 역할이 다릅니다.

In [3]:
y, groups = train.isFraud.astype('int8'), train.DT_M
oof, oof_scaled, pred = np.full(len(train), np.nan), np.full(len(train), np.nan), np.zeros(len(test))
importance, folds = np.zeros(len(features)), []
for fold, (fit_idx, val_idx) in enumerate(GroupKFold(FOLDS).split(train, y, groups)):
    assert set(groups.iloc[fit_idx]).isdisjoint(groups.iloc[val_idx])
    fit_x, val_x, test_x = train[features].iloc[fit_idx].copy(), train[features].iloc[val_idx].copy(), test[features].copy()
    for col, (train_key, test_key) in target_keys.items():
        if col in features:
            mapping = y.iloc[fit_idx].groupby(train_key.iloc[fit_idx]).mean()
            fit_x[col] = train_key.iloc[fit_idx].map(mapping).fillna(y.iloc[fit_idx].mean())
            val_x[col] = train_key.iloc[val_idx].map(mapping).fillna(y.iloc[fit_idx].mean())
            test_x[col] = test_key.map(mapping).fillna(y.iloc[fit_idx].mean())
    fit_x, val_x, test_x = [frame.replace([np.inf, -np.inf], np.nan) for frame in [fit_x, val_x, test_x]]
    model = CatBoostClassifier(**params, allow_writing_files=False)
    model.fit(fit_x, y.iloc[fit_idx], eval_set=(val_x, y.iloc[val_idx]), cat_features=categorical_features, use_best_model=True, verbose=False)
    values = model.predict_proba(val_x)[:, 1]
    pred += model.predict_proba(test_x)[:, 1] / FOLDS
    importance += model.feature_importances_ / FOLDS
    best_iteration = model.get_best_iteration()

    oof[val_idx] = values
    spread = values.max() - values.min()
    oof_scaled[val_idx] = (values - values.min()) / spread if spread else 0
    folds.append({'fold': fold, 'months': sorted(map(int, groups.iloc[val_idx].unique())),
                  'auc': float(roc_auc_score(y.iloc[val_idx], values)), 'best_iteration': int(best_iteration)})
    logger.info('fold=%d months=%s AUC=%.6f best_iteration=%d', fold, folds[-1]['months'], folds[-1]['auc'], best_iteration)
    del model, fit_x, val_x, test_x
    gc.collect()
assert np.isfinite(oof).all() and np.isfinite(pred).all()
assert np.all((oof >= 0) & (oof <= 1)) and np.all((pred >= 0) & (pred <= 1))
display(pd.DataFrame(folds))

fold=0 months=[12] AUC=0.810771 best_iteration=99


fold=1 months=[15] AUC=0.868992 best_iteration=98


fold=2 months=[13] AUC=0.856963 best_iteration=99


fold=3 months=[17] AUC=0.865869 best_iteration=83


fold=4 months=[14] AUC=0.892527 best_iteration=99


fold=5 months=[16] AUC=0.889058 best_iteration=55


,fold,months,auc,best_iteration
0,0,[12],0.810771,99
1,1,[15],0.868992,98
2,2,[13],0.856963,99
3,3,[17],0.865869,83
4,4,[14],0.892527,99
5,5,[16],0.889058,55


### fold 표에서 먼저 볼 부분
월 12의 AUC가 다른 여러 월보다 낮고, best_iteration은 일부 fold에서 99입니다. 0부터 세므로 100-tree 상한에 도달한 것입니다.
'100개가 최적'이라는 근거가 아니라 더 학습했을 때의 결과가 아직 없다는 뜻입니다. 월별 난이도 차이와 학습 상한을 따로 읽으세요.

## 4. 산출물과 변경

### 결과 파일은 다음 비교를 위한 실험 기록입니다

`oof.csv`에는 거래 번호·정답·월·OOF가 같이 있어 **어느 월/그룹에서 오답이 많은지** 다시 조사할 수 있습니다.
`pred_test.npy`는 fold 모델들의 test 평균이고, `submission.csv`는 sample_submission의 거래 ID 순서로 맞춰 저장합니다.
배열 위치가 같은 것처럼 보여도 ID 기준 확인을 해야 다른 데이터 순서에서 틀린 거래의 예측을 내지 않습니다. 축소 실행은 전체 제출 대신 sample_predictions를 저장합니다.

`model.feature_importances_`는 이 Logloss 모델에서 기본적으로 PredictionValuesChange를 제공합니다. 모델의 예측값 변화에 대한 중요도입니다.
다른 모델의 gain 중요도와 정의·스케일이 같지 않아 숫자를 직접 비교하지 않습니다. 높은 중요도는 인과관계나 제거 시 반드시 큰 성능 하락을 뜻하지도 않습니다.
비슷한 피처가 여럿이면 중요도를 나눠 갖거나 한 피처에 몰릴 수 있습니다. 제거 비교는 같은 검증에서 별도 수행해야 합니다.
[중요도 정의](https://catboost.ai/docs/en/concepts/fstr)에서 종류를 확인할 수 있습니다.

**아직 실행하지 않은 다음 실험:** 검증 고정 → UID 직접 포함/제외 → 지배값 flag 처리 포함/제외 → depth 6/8/10 비교 → rounds와 learning rate 조정.
UID 실험은 seen/unseen 그룹의 점수를 함께 보고, 미래 월 holdout에서도 다시 확인하면 특정 고객 기억에 기대는지 판단하기 좋습니다.
한 번에 여러 가공을 제거하면 무엇이 차이를 만들었는지 알 수 없으므로 하나씩 비교하세요.

In [4]:
np.save(OUT / 'oof_train.npy', oof)
np.save(OUT / 'oof_scaled.npy', oof_scaled)
np.save(OUT / 'pred_test.npy', pred)
pd.DataFrame({'TransactionID': train.TransactionID, 'isFraud': y, 'oof': oof, 'oof_source_scaled': oof_scaled, 'month': groups}).to_csv(OUT / 'oof.csv', index=False)
pd.Series(importance, index=features, name='importance').sort_values(ascending=False).to_csv(OUT / 'feature_importance.csv')
submission = pd.Series(pred, index=test.TransactionID, name='isFraud')
if NROWS:
    submission.to_csv(OUT / 'sample_predictions.csv')
else:
    sample = pd.read_csv(get_data_dir() / 'sample_submission.csv').TransactionID
    assert submission.index.is_unique and set(sample) == set(submission.index)
    submission.reindex(sample).to_csv(OUT / 'submission.csv')
metrics = {'scope': 'monthly sampled smoke' if NROWS else 'full raw CSV public baseline',
           'train_rows': len(train), 'test_rows': len(test), 'features': len(features), 'folds': folds, 'params': params,
           'rounds': ROUNDS, 'target_mean': 'fold-local', 'numeric_precision': 'float32, not source float16',
           'oof_auc': float(roc_auc_score(y, oof)), 'source_scaled_oof_auc': float(roc_auc_score(y, oof_scaled)),
           'elapsed_seconds': time.perf_counter() - started, 'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'catboost': catboost.__version__}}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
logger.info('OOF AUC=%.6f source-scaled AUC=%.6f elapsed=%.1fs', metrics['oof_auc'], metrics['source_scaled_oof_auc'], metrics['elapsed_seconds'])
display(metrics)

OOF AUC=0.857846 source-scaled AUC=0.858157 elapsed=42.8s


{'scope': 'monthly sampled smoke',
 'train_rows': 30000,
 'test_rows': 30002,
 'features': 720,
 'folds': [{'fold': 0,
   'months': [12],
   'auc': 0.810771139369903,
   'best_iteration': 99},
  {'fold': 1, 'months': [15], 'auc': 0.8689919427624346, 'best_iteration': 98},
  {'fold': 2, 'months': [13], 'auc': 0.8569630498453266, 'best_iteration': 99},
  {'fold': 3, 'months': [17], 'auc': 0.8658694978330775, 'best_iteration': 83},
  {'fold': 4, 'months': [14], 'auc': 0.8925266106442578, 'best_iteration': 99},
  {'fold': 5,
   'months': [16],
   'auc': 0.8890580338221425,
   'best_iteration': 55}],
 'params': {'iterations': 100,
  'learning_rate': 0.07,
  'eval_metric': 'AUC',
  'loss_function': 'Logloss',
  'random_seed': 42,
  'metric_period': 500,
  'od_wait': 500,
  'task_type': 'GPU',
  'depth': 8,
  'thread_count': 8},
 'rounds': 100,
 'target_mean': 'fold-local',
 'numeric_precision': 'float32, not source float16',
 'oof_auc': 0.8578456173905451,
 'source_scaled_oof_auc': 0.8581566

### 저장된 점수의 정확한 범위
월별 표본 30,000 train/30,002 test, 6-fold, 100 iterations, 720피처입니다. raw OOF AUC=.857846, source-scaled=.858157입니다.
두 점수의 차이는 위 fold별 min-max 때문이며 모델을 추가 학습한 개선이 아닙니다. 공개 best-single test 예측이나 최종 우승 성능을 재현한 점수로 읽지 않습니다.
**스스로 설명해보세요:** 빈도 숫자, UID 범주, target mean은 모두 그룹 정보를 쓰지만 각각 어떤 정보를 남기고 어떤 라벨을 사용하나요?